In [1]:
#Input
filtered_count_data = '../nf_output/data/Visium_FFPE_V43T08-051_D_filtered.csv'
spatial_data = '../nf_output/data/Visium_FFPE_V43T08-051_D_annotation.csv'
sample_name = 'Visium_FFPE_V43T08-051_D'
r_path = '/Library/Frameworks/R.framework/Resources'
min_clust = 11
max_clust = 15

In [2]:
import torch
import re
import os
import sys
sys.path.append('GraphST_clustering')
from graphST_clustering_functions import *
from GraphST.GraphST import GraphST

In [3]:
#Output
sample = os.path.basename(filtered_count_data)
parts = sample.split('_')
last_char = parts[-1].split('.')[0]
if last_char.isdigit() == True: #simulation
    seed = last_char
    graphST_output = f"{sample_name}_sim_{seed}_clustered.h5ad"
else:
    graphST_output = f"{sample_name}_ref_clustered.h5ad"

In [4]:
# Run device, by default, the package is implemented on 'cpu'. We recommend using GPU.
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda:1')
else:
    device = torch.device('cpu')

# the location of R, which is necessary for mclust algorithm. Please replace the path below with local R installation path
os.environ["R_HOME"] = r_path

In [5]:
data, col_label = data_preprocessing(filtered_count_data, spatial_data)

In [6]:
model = GraphST.GraphST(data, device=device, random_seed = 1)
# train model
data = model.train()

Begin to train ST data...


100%|██████████| 600/600 [00:36<00:00, 16.26it/s]

Optimization finished for ST data!


In [ ]:
# clustering 
methods = ['mclust','louvain', 'leiden']
data = graphST_methods_loop(data, methods, list(range(min_clust,max_clust+1)), start=0.01, end=2, refinement= True)

Searching resolution...
... for [11, 12, 13, 14, 15]
resolution=1.99, cluster number=16
resolution=1.98, cluster number=16
resolution=1.97, cluster number=16
resolution=1.96, cluster number=16
resolution=1.95, cluster number=15
save the resolution (1.95)
resolution=1.94, cluster number=16
resolution=1.93, cluster number=16
resolution=1.9200000000000002, cluster number=16
resolution=1.9100000000000001, cluster number=16
resolution=1.9000000000000001, cluster number=16
resolution=1.8900000000000001, cluster number=15
resolution=1.8800000000000001, cluster number=15
resolution=1.87, cluster number=16
resolution=1.86, cluster number=14
save the resolution (1.86)
resolution=1.85, cluster number=14
resolution=1.84, cluster number=13
save the resolution (1.84)
resolution=1.83, cluster number=13
resolution=1.82, cluster number=13
resolution=1.81, cluster number=13
resolution=1.8, cluster number=13
resolution=1.79, cluster number=13
resolution=1.78, cluster number=13
resolution=1.77, cluster nu

In [ ]:
data.write_h5ad(graphST_output)